In [1]:
# pip install xarray netcdf4

In [2]:
import xarray as xr
import numpy as np
import os
import glob
from functools import reduce

folder = "raw_data"
files = sorted(glob.glob(os.path.join(folder, "*.nc")))
print("Found", len(files), "files\n")

Found 9 files



In [3]:
# ------------------------- STEP 1: VARIABLE MAP -------------------------

print("VARIABLE MAP (file → variables):\n")

for f in files:
    ds = xr.open_dataset(f)
    vars_in_file = list(ds.data_vars.keys())
    print(f"{os.path.basename(f)}  →  {vars_in_file}")

VARIABLE MAP (file → variables):

chl.nc  →  ['chl']
currents.nc  →  ['uo', 'vo']
nutrients.nc  →  ['fe', 'no3', 'po4', 'si']
o2.nc  →  ['o2']
ph.nc  →  ['ph']
so.nc  →  ['so']
spco2.nc  →  ['spco2']
thetao.nc  →  ['thetao']
wo.nc  →  ['wo']


In [4]:
# ------------------------- STEP 2: PREPROCESS -------------------------

def preprocess(ds, filename):
    print(f"--- Preprocessing {filename} ---")

    # --------------------------------------------------
    # 0) ADD MISSING DEPTH FOR FILES LIKE spco2.nc
    # --------------------------------------------------
    if "depth" not in ds.coords:
        print(f"  -> depth missing. Adding depth = 0.5")
        ds = ds.expand_dims({"depth": [0.5]})

    # --------------------------------------------------
    # 1) ROUND COORDINATES
    # --------------------------------------------------
    ds = ds.assign_coords({
        "depth":     np.round(ds["depth"].values, 1),
        "latitude":  np.round(ds["latitude"].values, 2),
        "longitude": np.round(ds["longitude"].values, 2),
    })

    # TRUNCATE TIME TO DAY
    if "time" in ds.coords:
        new_time = np.array(ds["time"].values, dtype="datetime64[D]")
        ds = ds.assign_coords(time=new_time)

    # --------------------------------------------------
    # 2) CHECK FOR INVALID DEPTH VALUES (≠ 0.5)
    # --------------------------------------------------
    depth_values = ds["depth"].values
    invalid_depths = depth_values[depth_values != 0.5]

    if len(invalid_depths) > 0:
        print(f"  -> WARNING: Found {len(invalid_depths)} depth values not equal to 0.5:")
        print(f"       Invalid depths: {invalid_depths.tolist()}")
    else:
        print(f"  -> All depth values are valid (=0.5)")

    # --------------------------------------------------
    # 3) DETECT DUPLICATES
    # --------------------------------------------------
    df = ds.to_dataframe().reset_index()

    key_cols = ["time", "depth", "latitude", "longitude"]

    # Count duplicates
    dup_mask = df.duplicated(subset=key_cols, keep=False)
    dup_count = df[dup_mask].shape[0]

    print(f"  -> Duplicate coordinate entries after rounding: {dup_count}")

    # --------------------------------------------------
    # 4) AVERAGE OUT DUPLICATES
    # --------------------------------------------------
    df = df.groupby(key_cols).mean().reset_index()

    # Convert back to xarray
    ds_clean = df.set_index(key_cols).to_xarray()

    print(f"  -> After averaging, unique grid cells: {ds_clean.to_dataframe().shape[0]}")

    return ds_clean

In [5]:
# ------------------------- STEP 3: LOAD + PREPROCESS -------------------------

datasets = []
print("Preprocessing all files...")

for f in files:
    print("\n Opening:", os.path.basename(f))
    ds_clean = xr.open_dataset(f, chunks={"time": 1})

    ds_clean = preprocess(ds_clean, os.path.basename(f))

    datasets.append(ds_clean)

Preprocessing all files...

 Opening: chl.nc
--- Preprocessing chl.nc ---
  -> All depth values are valid (=0.5)
  -> Duplicate coordinate entries after rounding: 0
  -> After averaging, unique grid cells: 117572

 Opening: currents.nc
--- Preprocessing currents.nc ---
  -> All depth values are valid (=0.5)
  -> Duplicate coordinate entries after rounding: 0
  -> After averaging, unique grid cells: 964516

 Opening: nutrients.nc
--- Preprocessing nutrients.nc ---
  -> All depth values are valid (=0.5)
  -> Duplicate coordinate entries after rounding: 0
  -> After averaging, unique grid cells: 117572

 Opening: o2.nc
--- Preprocessing o2.nc ---
  -> All depth values are valid (=0.5)
  -> Duplicate coordinate entries after rounding: 0
  -> After averaging, unique grid cells: 117572

 Opening: ph.nc
--- Preprocessing ph.nc ---
  -> All depth values are valid (=0.5)
  -> Duplicate coordinate entries after rounding: 0
  -> After averaging, unique grid cells: 117572

 Opening: so.nc
--- Prep

In [6]:
# ------------------------- STEP 4: INNER JOIN COORDS -------------------------

def intersect_coord(coord):
    arrs = [ds[coord].values for ds in datasets]
    common = reduce(lambda a, b: np.intersect1d(a, b, assume_unique=True), arrs)
    base = datasets[0][coord].values
    mask = np.isin(base, common)
    return base[mask]


print("\nComputing INNER JOIN on coordinates...")

common_time  = intersect_coord("time")
common_depth = intersect_coord("depth")
common_lat   = intersect_coord("latitude")
common_lon   = intersect_coord("longitude")

print("\nAFTER INNER JOIN:")
print("  time:", len(common_time))
print("  depth:", len(common_depth))
print("  latitude:", len(common_lat))
print("  longitude:", len(common_lon))

if (
    len(common_time) == 0 or
    len(common_depth) == 0 or
    len(common_lat) == 0 or
    len(common_lon) == 0
):
    raise RuntimeError("Inner join produced empty coordinates! Aborting.")


Computing INNER JOIN on coordinates...

AFTER INNER JOIN:
  time: 532
  depth: 1
  latitude: 13
  longitude: 17


In [7]:
# ------------------------- STEP 5: SUBSET -------------------------

aligned = []
print("Subsetting every dataset to common coords...\n")

for i, ds in enumerate(datasets):
    fname = os.path.basename(files[i])
    print("Subsetting:", fname)
    ds2 = ds.sel(
        time=common_time,
        depth=common_depth,
        latitude=common_lat,
        longitude=common_lon,
        method=None
    )
    aligned.append(ds2)

Subsetting every dataset to common coords...

Subsetting: chl.nc
Subsetting: currents.nc
Subsetting: nutrients.nc
Subsetting: o2.nc
Subsetting: ph.nc
Subsetting: so.nc
Subsetting: spco2.nc
Subsetting: thetao.nc
Subsetting: wo.nc


In [8]:
# ------------------------- STEP 6: MERGE -------------------------

print("\nMerging all variables...\n")

merged = xr.merge(aligned, combine_attrs="override")

merged = merged.sortby("time")


Merging all variables...



In [9]:
# ------------------------- STEP 7: SAVE OUTPUT -------------------------

output = "data.nc"
print("Saving final merged dataset to:", output)

merged.to_netcdf(output)

print("\nDONE!")
print("Saved:", output)
print("\nFinal dataset summary:\n")
merged

Saving final merged dataset to: data.nc

DONE!
Saved: data.nc

Final dataset summary:



<xarray.Dataset> Size: 6MB
Dimensions:    (time: 532, depth: 1, latitude: 13, longitude: 17)
Coordinates:
  * time       (time) datetime64[s] 4kB 2022-06-01 2022-06-02 ... 2023-11-14
  * depth      (depth) float32 4B 0.5
  * latitude   (latitude) float32 52B 20.0 20.25 20.5 20.75 ... 22.5 22.75 23.0
  * longitude  (longitude) float32 68B 87.0 87.25 87.5 87.75 ... 90.5 90.75 91.0
Data variables: (12/13)
    chl        (time, depth, latitude, longitude) float32 470kB 0.286 ... nan
    uo         (time, depth, latitude, longitude) float32 470kB 0.1433 ... nan
    vo         (time, depth, latitude, longitude) float32 470kB 0.1213 ... nan
    fe         (time, depth, latitude, longitude) float32 470kB 0.001278 ... nan
    no3        (time, depth, latitude, longitude) float32 470kB 0.1593 ... nan
    po4        (time, depth, latitude, longitude) float32 470kB 0.0007956 ......
    ...         ...
    o2         (time, depth, latitude, longitude) float32 470kB 199.4 ... nan
    ph         (time, depth, latitude, longitude) float32 470kB 8.024 ... nan
    so         (time, depth, latitude, longitude) float32 470kB 32.92 ... nan
    spco2      (time, depth, latitude, longitude) float32 470kB 40.01 ... nan
    thetao     (time, depth, latitude, longitude) float32 470kB 30.36 ... nan
    wo         (time, depth, latitude, longitude) float32 470kB 5.102e-07 ......

In [10]:
import xarray as xr
import pandas as pd

# Load merged dataset
merged = xr.open_dataset("data.nc")

# Convert to DataFrame
df = merged.to_dataframe().reset_index()

df

,time,depth,latitude,longitude,chl,uo,vo,fe,no3,po4,si,o2,ph,so,spco2,thetao,wo
0,2022-06-01,0.5,20.0,87.00,0.285966,0.143302,0.121338,0.001278,0.159255,0.000796,2.166951,199.420868,8.024398,32.915531,40.012962,30.360138,5.101815e-07
1,2022-06-01,0.5,20.0,87.25,0.198817,0.285533,0.187017,0.001213,0.101606,0.000353,2.187757,198.929932,8.025283,32.908985,40.026024,30.338413,7.922670e-07
2,2022-06-01,0.5,20.0,87.50,0.157684,0.528991,0.245835,0.001165,0.072593,0.000246,2.220604,198.510590,8.026939,32.663116,39.930134,30.338757,8.813758e-07
3,2022-06-01,0.5,20.0,87.75,0.139504,0.651719,0.226462,0.001127,0.055083,0.000191,2.244844,198.271683,8.028164,32.484989,39.835117,30.413069,-7.822748e-07
4,2022-06-01,0.5,20.0,88.00,0.133310,0.715622,0.208896,0.001118,0.053426,0.000180,2.251250,198.161453,8.028683,32.405869,39.793114,30.456890,1.493642e-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117567,2023-11-14,0.5,23.0,90.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
117568,2023-11-14,0.5,23.0,90.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
117569,2023-11-14,0.5,23.0,90.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
117570,2023-11-14,0.5,23.0,90.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
df = df.dropna(how="any")

df

,time,depth,latitude,longitude,chl,uo,vo,fe,no3,po4,si,o2,ph,so,spco2,thetao,wo
0,2022-06-01,0.5,20.00,87.00,0.285966,0.143302,0.121338,0.001278,0.159255,0.000796,2.166951,199.420868,8.024398,32.915531,40.012962,30.360138,5.101815e-07
1,2022-06-01,0.5,20.00,87.25,0.198817,0.285533,0.187017,0.001213,0.101606,0.000353,2.187757,198.929932,8.025283,32.908985,40.026024,30.338413,7.922670e-07
2,2022-06-01,0.5,20.00,87.50,0.157684,0.528991,0.245835,0.001165,0.072593,0.000246,2.220604,198.510590,8.026939,32.663116,39.930134,30.338757,8.813758e-07
3,2022-06-01,0.5,20.00,87.75,0.139504,0.651719,0.226462,0.001127,0.055083,0.000191,2.244844,198.271683,8.028164,32.484989,39.835117,30.413069,-7.822748e-07
4,2022-06-01,0.5,20.00,88.00,0.133310,0.715622,0.208896,0.001118,0.053426,0.000180,2.251250,198.161453,8.028683,32.405869,39.793114,30.456890,1.493642e-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117486,2023-11-14,0.5,21.75,91.00,7.403145,0.021702,0.001965,0.008933,40.294735,0.017683,44.837063,297.890778,8.324100,3.306061,17.182795,26.817802,-6.295935e-08
117501,2023-11-14,0.5,22.00,90.50,5.534057,0.026373,0.002944,0.008240,17.276777,0.018899,18.837780,266.948181,8.206847,6.953091,16.379198,26.917717,1.127310e-06
117502,2023-11-14,0.5,22.00,90.75,7.723090,0.021342,-0.015882,0.010361,27.965033,0.023703,23.597303,298.528778,8.240985,5.706539,17.334682,26.825785,-2.428315e-07
117520,2023-11-14,0.5,22.25,91.00,11.135298,0.003299,-0.009229,0.011948,47.270050,0.091098,39.605762,366.201050,8.442244,0.786908,14.586455,26.777748,-1.573758e-07


In [12]:
# Check unique depth values
if "depth" in df.columns:
    unique_depths = df["depth"].unique()
    print("Unique depth values:", unique_depths)

    # If all depth values are exactly 0.5 → drop the whole column
    if len(unique_depths) == 1 and unique_depths[0] == 0.5:
        print("All depth values are 0.5 → dropping depth column.")
        df = df.drop(columns=["depth"])
    else:
        print("Depth column has multiple values → keeping it.")

df

Unique depth values: [0.5]
All depth values are 0.5 → dropping depth column.


,time,latitude,longitude,chl,uo,vo,fe,no3,po4,si,o2,ph,so,spco2,thetao,wo
0,2022-06-01,20.00,87.00,0.285966,0.143302,0.121338,0.001278,0.159255,0.000796,2.166951,199.420868,8.024398,32.915531,40.012962,30.360138,5.101815e-07
1,2022-06-01,20.00,87.25,0.198817,0.285533,0.187017,0.001213,0.101606,0.000353,2.187757,198.929932,8.025283,32.908985,40.026024,30.338413,7.922670e-07
2,2022-06-01,20.00,87.50,0.157684,0.528991,0.245835,0.001165,0.072593,0.000246,2.220604,198.510590,8.026939,32.663116,39.930134,30.338757,8.813758e-07
3,2022-06-01,20.00,87.75,0.139504,0.651719,0.226462,0.001127,0.055083,0.000191,2.244844,198.271683,8.028164,32.484989,39.835117,30.413069,-7.822748e-07
4,2022-06-01,20.00,88.00,0.133310,0.715622,0.208896,0.001118,0.053426,0.000180,2.251250,198.161453,8.028683,32.405869,39.793114,30.456890,1.493642e-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117486,2023-11-14,21.75,91.00,7.403145,0.021702,0.001965,0.008933,40.294735,0.017683,44.837063,297.890778,8.324100,3.306061,17.182795,26.817802,-6.295935e-08
117501,2023-11-14,22.00,90.50,5.534057,0.026373,0.002944,0.008240,17.276777,0.018899,18.837780,266.948181,8.206847,6.953091,16.379198,26.917717,1.127310e-06
117502,2023-11-14,22.00,90.75,7.723090,0.021342,-0.015882,0.010361,27.965033,0.023703,23.597303,298.528778,8.240985,5.706539,17.334682,26.825785,-2.428315e-07
117520,2023-11-14,22.25,91.00,11.135298,0.003299,-0.009229,0.011948,47.270050,0.091098,39.605762,366.201050,8.442244,0.786908,14.586455,26.777748,-1.573758e-07


In [13]:
# ------------------------------------------------------------
# CHECK FOR DUPLICATES USING PRIMARY KEY (time, lat, lon)
# ------------------------------------------------------------
pk = ["time", "latitude", "longitude"]

# Detect duplicates
dup_mask = df.duplicated(subset=pk, keep=False)
duplicate_rows = df[dup_mask]

if len(duplicate_rows) > 0:
    print("\nDUPLICATES FOUND!")
    print("Number of duplicate rows:", len(duplicate_rows))
    print("\nSample duplicates:")
    print(duplicate_rows.head())

    # --------------------------------------------------------
    # OPTION A — AVERAGE duplicates (recommended)
    # --------------------------------------------------------
    print("\nAveraging duplicate rows based on primary key...")
    df = df.groupby(pk).mean().reset_index()

else:
    print("\nNo duplicates found. Primary key is unique.")


No duplicates found. Primary key is unique.


In [14]:
df.to_csv("data.csv", index=False)